In [101]:
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPRegressor

import torch
import torch.nn as nn
import torch.optim as optim

In [102]:
DATA_DIR = Path("./initial_data")
HISTORY_DIR = Path("./history")

N_CANDIDATES = 100_000
SEED = 42

# False for minimisation; True for maximisation.
MAXIMISE = False

PROMISING_FRACTION = 0.30

# Hybrid acquisition weights.
WEIGHT_EI = 0.50
WEIGHT_LINEAR = 0.05
WEIGHT_SVM = 0.20
WEIGHT_MLP = 0.25

# Neural-network configuration.
MLP_HIDDEN_LAYERS = (64, 64)
MLP_MAX_ITER = 1500
MLP_ALPHA = 1e-3

# Gradient-guided search configuration.
GRADIENT_FRACTION = 0.30
GRADIENT_STEPS = 30
GRADIENT_STEP_SIZE = 0.03

# PyTorch config
TORCH_HIDDEN_LAYERS = (16, 16)

TORCH_EPOCHS = 500
TORCH_LEARNING_RATE = 0.001
TORCH_WEIGHT_DECAY = 0.001
TORCH_SEED = 42

In [103]:
def load_function(function_id, data_dir=DATA_DIR):
    path = data_dir / f"function_{function_id}"

    X = np.asarray(
        np.load(path / "initial_inputs.npy"), dtype=float
    )
    y = np.asarray(
        np.load(path / "initial_outputs.npy"), dtype=float
    ).reshape(-1)

    if X.ndim != 2 or len(X) != len(y):
        raise ValueError("Invalid input/output shapes.")

    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
        raise ValueError("Inputs and outputs must be finite.")

    if np.any(X < 0) or np.any(X > 1):
        raise ValueError("Inputs outside [0, 1].")

    return X, y

In [104]:
def load_history(function_id, history_dir=HISTORY_DIR):
    path = history_dir / f"function_{function_id}_history.csv"

    if not path.exists():
        return None, None

    df = pd.read_csv(path)

    x_columns = [
        column for column in df.columns
        if column.startswith("x")
    ]

    if "y" not in df.columns or not x_columns:
        raise ValueError(f"Invalid history file: {path}")

    # Sort x1, x2, ..., x10 numerically when possible.
    def column_order(name):
        suffix = name[1:]
        if suffix.isdigit():
            return (0, int(suffix))
        return (1, name)

    x_columns = sorted(x_columns, key=column_order)

    X = df[x_columns].to_numpy(dtype=float)
    y = df["y"].to_numpy(dtype=float)

    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
        raise ValueError(f"Non-finite values in {path}")

    if np.any(X < 0) or np.any(X > 1):
        raise ValueError(f"Inputs outside [0, 1] in {path}")

    return X, y

In [105]:
def load_all_data(function_id):
    X, y = load_function(function_id)
    X_history, y_history = load_history(function_id)

    if X_history is None:
        return X, y

    if X_history.shape[1] != X.shape[1]:
        raise ValueError(
            f"Function {function_id}: history and initial data "
            "have different numbers of dimensions."
        )

    X = np.vstack([X, X_history])
    y = np.concatenate([y, y_history])

    # Collapse exact duplicate inputs so repeated observations
    # do not receive extra training weight.
    df = pd.DataFrame(X)
    df["_y"] = y

    grouped = df.groupby(
        list(range(X.shape[1])),
        sort=False,
        as_index=False,
    )["_y"].mean()

    X = grouped.iloc[:, :X.shape[1]].to_numpy(dtype=float)
    y = grouped["_y"].to_numpy(dtype=float)

    return X, y

In [106]:
def fit_gp(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    dimension = X.shape[1]

    kernel = (
        ConstantKernel(1.0, constant_value_bounds="fixed")
        * Matern(
            length_scale=np.ones(dimension),
            length_scale_bounds="fixed",
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-5,
            noise_level_bounds="fixed",
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        optimizer=None,
        random_state=SEED,
    )

    model.fit(X_scaled, y)

    return model, scaler


In [107]:
def expected_improvement(X_candidates, model, scaler, y_best):
    X_scaled = scaler.transform(X_candidates)

    mean, std = model.predict(
        X_scaled,
        return_std=True,
    )

    std = np.maximum(std, 1e-12)

    improvement = y_best - mean
    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        + std * norm.pdf(z)
    )

    return np.maximum(ei, 0.0)

In [108]:
def fit_linear_model(X, y):
    model = make_pipeline(
        StandardScaler(),
        LinearRegression()
    )

    model.fit(X, y)

    return model

In [109]:
def fit_logistic_model(X, y):
    # Define the best 30% of observations as "promising"
    threshold = np.quantile(y, 0.30)

    labels = (y <= threshold).astype(int)

    # Logistic regression requires both classes.
    if len(np.unique(labels)) < 2:
        return None

    # With very small data sets, regularisation is important.
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=SEED,
        )
    )

    model.fit(X, labels)

    return model

In [110]:
def fit_svm_classifier(X, y):
    threshold = np.quantile(y, PROMISING_FRACTION)
    labels = (y <= threshold).astype(int)

    if len(np.unique(labels)) < 2:
        return None

    class_counts = np.bincount(labels, minlength=2)

    use_probability = class_counts.min() >= 5

    model = make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=use_probability,
            random_state=SEED,
        ),
    )

    model.fit(X, labels)

    return model, use_probability

In [111]:
def svm_promising_score(svm_result, candidates):
    if svm_result is None:
        return np.full(len(candidates), 0.5)

    model, use_probability = svm_result

    if use_probability:
        classes = model.named_steps["svc"].classes_
        positive_column = int(np.flatnonzero(classes == 1)[0])

        return model.predict_proba(candidates)[:, positive_column]

    decision = model.decision_function(candidates)
    decision = np.clip(decision, -30.0, 30.0)

    return 1.0 / (1.0 + np.exp(-decision))

In [112]:
def fit_mlp_surrogate(X, y, seed=TORCH_SEED):
    """Fit a regularised neural-network surrogate."""
    if len(y) < max(8, X.shape[1] + 2):
        return None, None

    if np.std(y) < 1e-12:
        return None, None

    torch.manual_seed(seed)

    # Standardise inputs and outputs
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_scaled = x_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(
        np.asarray(y).reshape(-1, 1)
    )

    X_tensor = torch.tensor(
        X_scaled, dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_scaled, dtype=torch.float32
    )

    # Build neural network architecture
    layers = []
    input_size = X.shape[1]

    for hidden_size in TORCH_HIDDEN_LAYERS:

        layers.append(
            nn.Linear(input_size, hidden_size)
        )

        layers.append(nn.ReLU())

        input_size = hidden_size

    # Single output neuron for regression
    layers.append(nn.Linear(input_size, 1))

    model = nn.Sequential(*layers)

    # Optimiser and loss function
    optimiser = optim.Adam(
        model.parameters(),
        lr=TORCH_LEARNING_RATE,
        weight_decay=TORCH_WEIGHT_DECAY
    )

    loss_function = nn.MSELoss()

    # Training loop
    model.train()

    for epoch in range(TORCH_EPOCHS):

        optimiser.zero_grad()

        predictions = model(X_tensor)

        loss = loss_function(
            predictions,
            y_tensor
        )

        loss.backward()

        optimiser.step()

    model.eval()

    return model, (x_scaler, y_scaler)

In [113]:
def mlp_predict(model_result, X):
    """Return predictions on the original objective scale."""
    model, scalers = model_result

    if model is None:
        return None

    x_scaler, y_scaler = scalers

    X_scaled = x_scaler.transform(X)

    X_tensor = torch.tensor(
        X_scaled,
        dtype=torch.float32
    )

    model.eval()

    with torch.no_grad():

        predictions = model(X_tensor).numpy()

    predictions = y_scaler.inverse_transform(
        predictions
    ).ravel()

    return predictions

In [114]:
def mlp_input_gradient(model_result, X):
    """Calculate d(predicted objective)/dX for a fitted ReLU MLP."""
    model, scalers = model_result

    if model is None:
        return None

    x_scaler, y_scaler = scalers

    X = np.atleast_2d(
        np.asarray(X, dtype=float)
    )

    X_scaled = x_scaler.transform(X)

    X_tensor = torch.tensor(
        X_scaled,
        dtype=torch.float32,
        requires_grad=True
    )

    model.eval()

    predictions = model(X_tensor)

    gradients = torch.autograd.grad(
        predictions.sum(),
        X_tensor
    )[0]

    gradients = gradients.detach().numpy()

    # Convert gradients to original input/output scales
    gradients *= float(y_scaler.scale_[0])

    gradients /= x_scaler.scale_.reshape(1, -1)

    return gradients

In [115]:
def generate_gradient_candidates(
    model_result,
    X,
    y,
    n_candidates,
    rng,
):
    """Generate local-search candidates using MLP gradients.
    Assumes y is already converted to a minimisation objective.
    """

    dimension = X.shape[1]

    if model_result[0] is None or n_candidates <= 0:
        return np.empty((0, dimension))

    n_starts = min(
        max(4, int(np.sqrt(n_candidates))),
        len(X),
    )

    # Start from the best observed points.
    start_indices = np.argsort(y)[:n_starts]
    starts = X[start_indices].copy()

    points = []

    for start in starts:
        x = start.copy()

        for _ in range(GRADIENT_STEPS):
            gradient = mlp_input_gradient(
                model_result,
                x.reshape(1, -1),
            )

            if gradient is None or not np.all(np.isfinite(gradient)):
                break

            gradient = gradient[0]
            gradient_norm = np.linalg.norm(gradient)

            if gradient_norm < 1e-10:
                break

            # Normalise the gradient to control the step length.
            x = np.clip(
                x - GRADIENT_STEP_SIZE
                * gradient / gradient_norm,
                0.0,
                1.0,
            )

            points.append(x.copy())

            if len(points) >= n_candidates:
                break

        if len(points) >= n_candidates:
            break

    # Preserve candidate count if gradients produce too few points.
    if len(points) < n_candidates:
        remaining = n_candidates - len(points)

        supplement = rng.uniform(
            0.0,
            1.0,
            size=(remaining, dimension),
        )

        points.extend(supplement)

    return np.asarray(points[:n_candidates])

In [116]:
def generate_candidates(n_candidates, dimension, rng):
    return rng.uniform(
        0.0,
        1.0, 
        size=(n_candidates, dimension)
    )

In [117]:
def minmax_scale(values):
    values = np.asarray(values, dtype=float)

    low = np.min(values)
    high = np.max(values)

    if high - low < 1e-12:
        return np.zeros_like(values)

    return (values - low) / (high - low)

In [118]:
def propose_next(
    X,
    y,
    maximise=False,
    n_candidates=N_CANDIDATES,
    seed=SEED,
):
    if X.ndim != 2 or len(X) != len(y):
        raise ValueError("X and y have incompatible shapes.")

    if len(y) < 2:
        raise ValueError("At least two observations are required.")

    if n_candidates < 1:
        raise ValueError("n_candidates must be positive.")

    rng = np.random.default_rng(seed)

    # Convert maximisation into minimisation.
    target = -np.asarray(y, dtype=float) if maximise else np.asarray(y, dtype=float)

    # Fit all surrogate models.
    gp, gp_scaler = fit_gp(X, target)
    linear_model = fit_linear_model(X, target)
    svm_result = fit_svm_classifier(X, target)
    mlp_result = fit_mlp_surrogate(X, target, seed=seed)

    # Generate a mixed candidate pool.
    n_gradient = (
        int(n_candidates * GRADIENT_FRACTION)
        if mlp_result[0] is not None
        else 0
    )

    n_random = n_candidates - n_gradient

    random_candidates = generate_candidates(
        n_random,
        X.shape[1],
        rng,
    )

    if n_gradient > 0:
        gradient_candidates = generate_gradient_candidates(
            mlp_result,
            X,
            target,
            n_gradient,
            rng,
        )

        candidates = np.vstack([
            random_candidates,
            gradient_candidates,
        ])
    else:
        candidates = random_candidates

    # Remove candidates already evaluated.
    tree = cKDTree(X)
    distances, _ = tree.query(candidates, k=1)

    candidates = candidates[distances > 1e-10]

    if len(candidates) == 0:
        raise RuntimeError("No candidates remain after filtering.")

    # 1. Gaussian Process Expected Improvement.
    y_best = np.min(target)

    ei = expected_improvement(
        candidates,
        gp,
        gp_scaler,
        y_best,
    )

    ei_score = minmax_scale(ei)

    # 2. Linear model: prefer lower predicted objective.
    linear_prediction = linear_model.predict(candidates)

    linear_score = minmax_scale(
        np.max(linear_prediction) - linear_prediction
    )

    # 3. SVM: prefer candidates classified as promising.
    svm_score = svm_promising_score(
        svm_result,
        candidates,
    )

    # 4. MLP: prefer lower predicted objective.
    mlp_prediction = mlp_predict(
        mlp_result,
        candidates,
    )

    if mlp_prediction is None:
        mlp_score = np.zeros(len(candidates))
        mlp_weight = 0.0
    else:
        mlp_score = minmax_scale(
            np.max(mlp_prediction) - mlp_prediction
        )
        mlp_weight = WEIGHT_MLP

    # Normalise active weights.
    weights = np.array([
        WEIGHT_EI,
        WEIGHT_LINEAR,
        WEIGHT_SVM,
        mlp_weight,
    ], dtype=float)

    if np.any(weights < 0) or weights.sum() <= 0:
        raise ValueError("Acquisition weights are invalid.")

    weights /= weights.sum()

    # Combined acquisition score.
    score = (
        weights[0] * ei_score
        + weights[1] * linear_score
        + weights[2] * svm_score
        + weights[3] * mlp_score
    )

    best_idx = int(np.argmax(score))

    return (
        candidates[best_idx],
        gp,
        gp_scaler,
        linear_model,
        svm_result,
        mlp_result,
        float(score[best_idx]),
    )

In [119]:
def format_query(x):
    return "-".join(
        f"{value:.6f}"
        for value in x
    )

In [120]:
query_results = []

for i in range(1, 9):
    print("\n" + "=" * 50)
    print(f"FUNCTION {i}")
    print("=" * 50)

    try:
        X, y = load_all_data(i)

        print(f"Number of observations: {len(y)}")

        if len(y) != 14:
            print(
                f"WARNING: Expected 14 observations, "
                f"but found {len(y)}."
            )

        # Identify current best observation
        if MAXIMISE:
            best_idx = np.argmax(y)
        else:
            best_idx = np.argmin(y)

        best_x = X[best_idx]
        best_y = y[best_idx]

        print(f"Current best output: {best_y:.6f}")

        # Generate next query
        (
            x_next,
            gp,
            gp_scaler,
            linear_model,
            svm_result,
            mlp_result,
            score
        ) = propose_next(
            X,
            y,
            maximise=MAXIMISE,
            n_candidates=N_CANDIDATES,
            seed=SEED + i
        )

        # Format query for submission
        query = format_query(x_next)

        print("\nCurrent best input:")
        print(best_x)

        print("\nRecommended next input:")
        print(x_next)

        print("\nFormatted query:")
        print(query)

        print(f"\nAcquisition score: {score:.6f}")

        print(
            "\nSVM:",
            "Fitted" if svm_result is not None
            else "Unavailable"
        )

        print(
            "MLP:",
            "Fitted" if mlp_result[0] is not None
            else "Unavailable"
        )

        query_results.append({
            "function": i,
            "observations": len(y),
            "best_output": best_y,
            "query": query,
            "acquisition_score": score
        })

    except Exception as exc:
        print(f"ERROR: Function {i} failed: {exc}")


# ==========================================
# SAVE RESULTS
# ==========================================

if query_results:
    results_df = pd.DataFrame(query_results)

    print("\n" + "=" * 50)
    print("QUERY SUMMARY")
    print("=" * 50)

    print(
        results_df[
            ["function", "observations", "best_output", "query"]
        ].to_string(index=False)
    )

else:
    print("No queries were successfully generated.")

    print("\n====================================")
    print(f"FUNCTION {i}")
    print("====================================")
    print("Observations:        ", X.shape[0])
    print("Dimensions:          ", X.shape[1])
    print("Current best x:      ", best_x)
    print("Current best y:      ", f"{best_y:.6f}")
    print(
        "SVM status:          ",
        "fitted" if svm_result is not None else "unavailable",
    )
    print(
        "MLP status:          ",
        "fitted" if mlp_result[0] is not None
        else "not fitted",
    )
    print("Proposed x:          ", x_next)
    print("Acquisition score:   ", f"{score:.6f}")
    print("Query:               ", format_query(x_next))


FUNCTION 1
Number of observations: 14
Current best output: -0.003606


/home/arwel/Projects/icl/black-box-capstone/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Current best input:
[0.65011406 0.68152635]

Recommended next input:
[0.50818001 0.55775083]

Formatted query:
0.508180-0.557751

Acquisition score: 0.887087

SVM: Fitted
MLP: Fitted

FUNCTION 2
Number of observations: 14
Current best output: -0.065624


/home/arwel/Projects/icl/black-box-capstone/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Current best input:
[0.142699 0.349005]

Recommended next input:
[0.04293073 0.00024893]

Formatted query:
0.042931-0.000249

Acquisition score: 0.918381

SVM: Fitted
MLP: Fitted

FUNCTION 3
Number of observations: 19
Current best output: -0.444847


/home/arwel/Projects/icl/black-box-capstone/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


... (7 outputs left out)

/home/arwel/Projects/icl/black-box-capstone/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Current best input:
[0.573815 0.569763 0.043645 0.194446 0.976853 0.679647]

Recommended next input:
[0.21189692 0.35466155 0.45186687 0.83437511 0.98756582 0.89124746]

Formatted query:
0.211897-0.354662-0.451867-0.834375-0.987566-0.891247

Acquisition score: 0.900781

SVM: Fitted
MLP: Fitted

FUNCTION 8
Number of observations: 44
Current best output: 5.246584


/home/arwel/Projects/icl/black-box-capstone/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Current best input:
[1.       0.784261 1.       0.260685 0.538566 0.199064 0.596507 0.299898]

Recommended next input:
[1.         0.82827545 1.         0.14914969 0.54154197 0.26101295
 0.60942642 0.28156342]

Formatted query:
1.000000-0.828275-1.000000-0.149150-0.541542-0.261013-0.609426-0.281563

Acquisition score: 0.939348

SVM: Fitted
MLP: Fitted

QUERY SUMMARY
 function  observations  best_output                                                                   query
        1            14    -0.003606                                                       0.508180-0.557751
        2            14    -0.065624                                                       0.042931-0.000249
        3            19    -0.444847                                              0.045795-0.391136-0.995832
        4            34   -44.346413                                     0.999494-0.997497-0.979596-0.521434
        5            24     0.112940                                     0.753980-0.7